# Single-modality baseline model

### Preparation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.model_selection import KFold

In [2]:
df = pd.read_pickle(r"C:\Users\Juli\Documents\Master\Projekt Genomforschung\projekt_genomforschung\harmonized_data.pkl")
df.head()

,SequencingID,ModelID,TSPAN6 (7105),SCYL3 (57147),BAD (572),LAP3 (51056),SNX11 (29916),CASP10 (843),CFLAR (8837),FKBP4 (2288),...,Bit_1014,Bit_1015,Bit_1016,Bit_1017,Bit_1018,Bit_1019,Bit_1020,Bit_1021,Bit_1022,Bit_1023
0,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,0,0,0,0,0,0,0,0
1,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,0,1,0,1,0,0,0,0
2,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,0,0,0,0,0,0,0,0
3,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,0,0,1,1,0,0,0,0
4,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,0,1,0,1,0,0,0,0


In [24]:
l1000_genes = df.iloc[:, 2:980].values

## Genomic features only

### Random Forest

In [ ]:
X = l1000_genes # L1000 landmark genes (genomic features)
y = df['AUC'].values # y: drug response
groups = df['ModelID'].values # groups: cell line identifiers (to ensure held-out validation)

# initialize the Random Forest Regressor
rf_genomic = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

# setup GroupKFold for held-out cell line validation
gkf = GroupKFold(n_splits=5)

# perform Cross-Validation
print("Starting Cross-Validation...")
cv_results = cross_validate(
    rf_genomic, X, y, 
    groups=groups, 
    cv=gkf,
    scoring=['neg_mean_squared_error', 'r2'],
    return_train_score=True
)

# output Results
mse_scores = -cv_results['test_neg_mean_squared_error']
r2_scores = cv_results['test_r2']

print(f"Mean MSE: {np.mean(mse_scores):.4f} (+/- {np.std(mse_scores):.4f})")
print(f"Mean R2 Score: {np.mean(r2_scores):.4f}")

# feature Importance
# fit once on the full set to see which genes drive the prediction
rf_genomic.fit(X, y)


Starting Cross-Validation...
Mean MSE: 0.0206 (+/- 0.0007)
Mean R2 Score: 0.0184

Top 5 Genetic Features:


In [13]:
importances = pd.Series(rf_genomic.feature_importances_, index=df.iloc[:, 2:980].columns.values)
print("\nTop 5 Genetic Features:")
print(importances.sort_values(ascending=False).head(5))



Top 5 Genetic Features:
TJP1 (7082)      0.315030
SQSTM1 (8878)    0.016814
UBE3B (89910)    0.009887
GTF2E2 (2961)    0.008869
NOLC1 (9221)     0.008650
dtype: float64


### Random Forest restricted to one drug

In [30]:
# choose single drug for detailed analysis
top_drugs = df['DRUG_ID'].value_counts()
print("Top Drugs by frequency:")
print(top_drugs.head(5))

best_drug_id = top_drugs.index[0]
print(f"\nTesting with DrugID: {best_drug_id}")

df_single_drug = df[df['DRUG_ID'] == best_drug_id]

# prepare features X and target y for this single drug
X_single = df_single_drug.iloc[:, 2:980].values
y_single = df_single_drug['AUC'].values
groups_single = df_single_drug['ModelID'].values

# initialize a new Random Forest for this single drug
rf_single = RandomForestRegressor(
    n_estimators=100, 
    max_depth=15, 
    n_jobs=-1, 
    random_state=42
)

# use GroupKFold for this single drug as well to ensure we are not overfitting to specific cell lines
cv_results_single = cross_validate(
    rf_single, X_single, y_single, 
    groups=groups_single, 
    cv=gkf,
    scoring=['neg_mean_squared_error', 'r2'],
    return_train_score=True
)

print(f"Single Drug R2 Score: {np.mean(cv_results_single['test_r2']):.4f}")

Top Drugs by frequency:
DRUG_ID
1862    736
1003    735
1034    735
1047    735
1053    735
Name: count, dtype: int64

Testing with DrugID: 1862
Single Drug R2 Score: 0.1877


In [31]:
rf_single.fit(X_single, y_single)
importances_single = pd.Series(rf_single.feature_importances_, index=df_single_drug.iloc[:, 2:980].columns.values)
print("\nTop 5 Genetic Features for Single Drug:")
print(importances_single.sort_values(ascending=False).head(5))


Top 5 Genetic Features for Single Drug:
TSKU (25987)      0.029049
FBXL12 (54850)    0.023325
APP (351)         0.022270
PTK2 (5747)       0.016196
IKZF1 (10320)     0.014000
dtype: float64


## Chemical features only

In [ ]:
import pandas as pd

# Angenommen, deine Spalte mit den Dictionaries heißt 'Pharma_Dict'
# df ist dein aktueller DataFrame

# 1. Wandle die Spalte in eine Liste von Dictionaries um und mache einen neuen DataFrame daraus
expanded_features = pd.DataFrame(df['PharmacophoreFeatures'].tolist())

# 2. Füge die neuen Spalten an deinen ursprünglichen DataFrame an 
# und lösche die alte, verschachtelte Spalte
df = pd.concat([df.drop('PharmacophoreFeatures', axis=1), expanded_features], axis=1)

# Optional: Fehlende Werte mit 0 auffüllen 
# (Falls ein Medikament z.B. keinen 'Donor' hat, steht dort sonst NaN)
df = df.fillna(0) 

print(df.head())

## Random forest

In [36]:
df.head()

,SequencingID,ModelID,TSPAN6 (7105),SCYL3 (57147),BAD (572),LAP3 (51056),SNX11 (29916),CASP10 (843),CFLAR (8837),FKBP4 (2288),...,Bit_1022,Bit_1023,Donor,Acceptor,Aromatic,Hydrophobe,LumpedHydrophobe,PosIonizable,NegIonizable,ZnBinder
0,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,1.0,5.0,3.0,5.0,1.0,0.0,0.0,0.0
1,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,5.0,9.0,3.0,10.0,2.0,2.0,0.0,0.0
2,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,4.0,6.0,1.0,0.0,0.0,0.0,0.0,0.0
4,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,5.0,14.0,2.0,7.0,3.0,0.0,0.0,0.0


In [ ]:
X_chem = df[['MorganFP', 'Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']].values # chemical features, i.e. Morgan fingerprints and pharmacophore features
y_chem = df['AUC'].values # y: drug response
groups = df['ModelID'].values # groups: cell line identifiers (to ensure held-out validation)

# initialize the Random Forest Regressor
rf_chemical = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

# setup GroupKFold for held-out cell line validation
gkf = GroupKFold(n_splits=5)

# perform Cross-Validation
print("Starting Cross-Validation...")
cv_chemical = cross_validate(
    rf_chemical, X_chem, y_chem, 
    groups=groups, 
    cv=gkf,
    scoring=['neg_mean_squared_error', 'r2'],
    return_train_score=True
)

# output Results
mse_scores = -cv_chemical['test_neg_mean_squared_error']
r2_scores = cv_chemical['test_r2']

print(f"Mean MSE: {np.mean(mse_scores):.4f} (+/- {np.std(mse_scores):.4f})")
print(f"Mean R2 Score: {np.mean(r2_scores):.4f}")

# feature Importance
# fit once on the full set to see which genes drive the prediction
rf_chemical.fit(X_chem, y_chem)
importances_c = pd.Series(rf_chemical.feature_importances_, index=df.iloc[:, 2:980].columns.values)
print("\nTop 5 Chemical Features:")
print(importances_c.sort_values(ascending=False).head(5))

In [65]:
# choose single drug for detailed analysis
top_cl = df['ModelID'].value_counts().index[0]
print("Top Cell Line by frequency:")
print(top_cl)

df_single_cell = df[df['ModelID'] == top_cl]

# prepare features X and target y for this single drug
morgan_cols = df_single_cell.columns[df_single_cell.columns.str.startswith('Bit_')]
X_single_c = df_single_cell.loc[:, morgan_cols]  + df_single_cell.loc[:, ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe',
                           'PosIonizable', 'NegIonizable', 'ZnBinder']] # chemical features, i.e. Morgan fingerprints and pharmacophore features
y_single_c = df_single_cell['AUC'].values

# initialize a new Random Forest for this single drug
rf_single_c = RandomForestRegressor(
    n_estimators=100, 
    max_depth=15, 
    n_jobs=-1, 
    random_state=42
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results_single = cross_validate(
    rf_single_c, X_single_c, y_single_c, 
    cv=kf,
    scoring=['neg_mean_squared_error', 'r2'],
    return_train_score=True
)

print(f"Single drug R2 Score: {np.mean(cv_results_single['test_r2']):.4f}")
rf_single_c.fit(X_single_c, y_single_c)
importances_single = pd.Series(rf_single_c.feature_importances_, index= X_single_c.columns)
print("\nTop 5 drug Features for Single Drug:")
print(importances_single.sort_values(ascending=False).head(5))

Top Cell Line by frequency:
ACH-000672
Single drug R2 Score: -0.0078

Top 5 drug Features for Single Drug:
Bit_924    0.014964
Bit_994    0.014943
Bit_870    0.014781
Bit_992    0.013715
Bit_853    0.012592
dtype: float64
